## 트랜스포머

### 데이터 준비

In [ ]:
from urllib import request
request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename="ChatBotData.csv")

import pandas as pd 
df = pd.read_csv("/kaggle/working/ChatBotData.csv")


### 데이터 전처리

In [ ]:
import re 
def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    return sentence



questions = df['Q'].apply(preprocess_sentence).tolist()
answers = df['A'].apply(preprocess_sentence).tolist()



import tensorflow_datasets as tfds
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    questions + answers, target_vocab_size=2**13)



START_TOKEN = [tokenizer.vocab_size]
END_TOKEN = [tokenizer.vocab_size + 1]
VOCAB_SIZE = tokenizer.vocab_size + 2
MAX_LENGTH = 40



def pad_seq(seq, maxlen=MAX_LENGTH):
    if len(seq) < maxlen:
        return seq + [0] * (maxlen - len(seq))
    return seq[:maxlen]



def tokenize_and_filter(inputs, outputs):
    tokenized_inputs, tokenized_outputs = [], []
    for (sentence1, sentence2) in zip(inputs, outputs):
        sentence1 = START_TOKEN + tokenizer.encode(sentence1) + END_TOKEN
        sentence2 = START_TOKEN + tokenizer.encode(sentence2) + END_TOKEN
        tokenized_inputs.append(sentence1)
        tokenized_outputs.append(sentence2)
    tokenized_inputs = [pad_seq(seq) for seq in tokenized_inputs]
    tokenized_outputs = [pad_seq(seq) for seq in tokenized_outputs]
    return np.array(tokenized_inputs), np.array(tokenized_outputs)



questions, answers = tokenize_and_filter(questions, answers)



from torch.utils.data import Dataset, DataLoader



import torch
class ChatbotDataset(Dataset):
    def __init__(self, question, answer):
        self.question = question
        self.answer  = answer 

    def __len__(self):
        return len(self.question)

    def __getitem__(self, idx):
        return (torch.tensor(self.question[idx], dtype=torch.long), 
                torch.tensor(self.answer[idx][:-1], dtype=torch.long),
                torch.tensor(self.answer[idx][1:], dtype=torch.long))


BATCH_SIZE = 64
data = ChatbotDataset(questions, answers)
dataloader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)
test = next(iter(dataloader))
